# Colab Training - Hybrid BiLSTM-CRF POS Tagger

Notebook ini menjalankan eksperimen BiLSTM-CRF di Colab GPU.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import subprocess
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_PARENT = DRIVE_ROOT / 'POSTagPantun'
REPO_URL = 'https://github.com/NazirArifin/pantunify.git'
REPO_DIR = PROJECT_PARENT / 'pantunify'
PROJECT_PARENT.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repo already exists, skip clone.')

%cd {REPO_DIR}
!git --no-pager status -sb


/content/drive/MyDrive/POSTagPantun/pantunify
## main...origin/main


In [ ]:
!python -V
!pip -q install --upgrade pip
!pip -q install torch pytorch-crf


Python 3.12.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.7 MB/s eta 0:00:00


In [4]:
import json

DATA_DIR = REPO_DIR / 'data' / 'article5'
TRAIN_GSD = DATA_DIR / 'UD_Indonesian-GSD' / 'id_gsd-ud-train.conllu'
DEV_GSD = DATA_DIR / 'UD_Indonesian-GSD' / 'id_gsd-ud-dev.conllu'
TRAIN_CSUI = DATA_DIR / 'id_csui-ud-train.conllu'
OUT_DIR = DATA_DIR / 'processed_crf'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [TRAIN_GSD, DEV_GSD, TRAIN_CSUI]:
    print(p, '->', 'OK' if p.exists() else 'MISSING')


/content/drive/MyDrive/POSTagPantun/pantunify/data/article5/UD_Indonesian-GSD/id_gsd-ud-train.conllu -> OK
/content/drive/MyDrive/POSTagPantun/pantunify/data/article5/UD_Indonesian-GSD/id_gsd-ud-dev.conllu -> OK
/content/drive/MyDrive/POSTagPantun/pantunify/data/article5/id_csui-ud-train.conllu -> OK


In [5]:
CONVERTER = REPO_DIR / 'scripts' / 'article5' / 'convert_conllu_to_jsonl.py'
!python {CONVERTER} \
  --train-conllu {TRAIN_GSD} {TRAIN_CSUI} \
  --dev-conllu {DEV_GSD} \
  --out-dir {OUT_DIR}


[OK] train jsonl: /content/drive/MyDrive/POSTagPantun/pantunify/data/article5/processed_crf/train.jsonl
[OK] dev jsonl: /content/drive/MyDrive/POSTagPantun/pantunify/data/article5/processed_crf/dev.jsonl
[OK] vocab: /content/drive/MyDrive/POSTagPantun/pantunify/data/article5/processed_crf/vocab.json
[INFO] train_sentences=5138, dev_sentences=559
[INFO] word_vocab=20416, char_vocab=281, tags=17


In [6]:
VOCAB_PATH = OUT_DIR / 'vocab.json'
with open(VOCAB_PATH, 'r', encoding='utf-8') as f:
    vocab = json.load(f)

num_chars = len(vocab['char_vocab'])
vocab_size = len(vocab['word_vocab'])
num_classes = len(vocab['tag_vocab'])
print(num_chars, vocab_size, num_classes)


281 20416 17


In [7]:
TRAIN_SCRIPT = REPO_DIR / 'scripts' / 'article5' / 'train_hybrid_bilstm_crf_pos.py'
TRAIN_JSONL = OUT_DIR / 'train.jsonl'
DEV_JSONL = OUT_DIR / 'dev.jsonl'
CKPT_DIR = REPO_DIR / 'scripts' / 'article5' / 'checkpoints_crf_colab'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

!python {TRAIN_SCRIPT} \
  --train-jsonl {TRAIN_JSONL} \
  --valid-jsonl {DEV_JSONL} \
  --num-chars {num_chars} \
  --vocab-size {vocab_size} \
  --num-classes {num_classes} \
  --save-dir {CKPT_DIR} \
  --epochs 20 \
  --batch-size 64 \
  --word-emb-dim 300


Traceback (most recent call last):
  File "/content/drive/MyDrive/POSTagPantun/pantunify/scripts/article5/train_hybrid_bilstm_crf_pos.py", line 10, in <module>
    from HybridBiLSTMCRFPOSTagger import HybridBiLSTMCRFPOSTagger
  File "/content/drive/MyDrive/POSTagPantun/pantunify/scripts/article5/HybridBiLSTMCRFPOSTagger.py", line 3, in <module>
    from torchcrf import CRF
ModuleNotFoundError: No module named 'torchcrf'


In [ ]:
import torch

last_ckpt = CKPT_DIR / 'last.pt'
best_ckpt = CKPT_DIR / 'best.pt'
print('last exists:', last_ckpt.exists())
print('best exists:', best_ckpt.exists())

if last_ckpt.exists():
    obj = torch.load(last_ckpt, map_location='cpu')
    print('[LAST] epoch:', obj.get('epoch'))
    print('[LAST] metrics:', obj.get('metrics'))
if best_ckpt.exists():
    obj = torch.load(best_ckpt, map_location='cpu')
    print('[BEST] epoch:', obj.get('epoch'))
    print('[BEST] metrics:', obj.get('metrics'))
